# W1 Day4 (04/10 周四): PyTorch Module + 优化器 + 数据 + AMP

## 学习目标
- **理论 (1-1.5h)**: nn.Module / hook / state_dict; Adam / AdamW 原理; Dataset / DataLoader; AMP
- **实践 (2h)**: hook 可视化 + Adam vs SGD 对比 + AMP 训练 ResNet on CIFAR-10
- **产出物**: 优化器 + AMP 实验 notebook

---

## 目录
1. [nn.Module 深入理解](#1)
2. [Hook 机制: forward_hook / backward_hook](#2)
3. [state_dict: 模型保存与加载](#3)
4. [优化器原理: SGD → Momentum → Adam → AdamW](#4)
5. [Adam vs SGD 对比实验](#5)
6. [Dataset 与 DataLoader](#6)
7. [AMP 混合精度训练](#7)
8. [综合实战: AMP 训练 ResNet on CIFAR-10](#8)
9. [学习率调度器](#9)
10. [思考题 & 总结](#10)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from torch.cuda.amp import autocast, GradScaler
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import time
import copy
from collections import OrderedDict

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

---
## 1. nn.Module 深入理解 <a id='1'></a>

### 1.1 Module 的核心机制

`nn.Module` 是所有神经网络模块的基类，提供:
- **参数管理**: `parameters()`, `named_parameters()`
- **子模块管理**: `children()`, `modules()`, `named_modules()`
- **设备/dtype 切换**: `to()`, `cuda()`, `cpu()`, `half()`
- **训练/推理切换**: `train()`, `eval()` (影响 Dropout, BN)
- **序列化**: `state_dict()`, `load_state_dict()`

In [ ]:
print("=" * 60)
print("nn.Module 内部机制")
print("=" * 60)

class MyNet(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        # 子模块: 自动注册到 _modules
        self.layer1 = nn.Linear(in_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, out_dim)
        self.dropout = nn.Dropout(0.3)
        
        # 自定义参数: 用 nn.Parameter 注册
        self.scale = nn.Parameter(torch.ones(1))  # 可训练
        
        # 普通 tensor: 不会被 parameters() 返回
        self.register_buffer('running_mean', torch.zeros(hidden_dim))  # buffer: 保存但不训练
    
    def forward(self, x):
        x = self.layer1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.layer2(x)
        return x * self.scale

net = MyNet(10, 32, 5)

# 参数遍历
print("\n--- named_parameters() ---")
for name, param in net.named_parameters():
    print(f"  {name:25s}: shape={str(list(param.shape)):15s} requires_grad={param.requires_grad}")

total_params = sum(p.numel() for p in net.parameters())
trainable = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(f"\n  总参数量: {total_params:,} (可训练: {trainable:,})")

In [ ]:
# 模块遍历
print("--- named_modules() (递归) ---")
for name, module in net.named_modules():
    print(f"  {'(root)' if name == '' else name:25s}: {module.__class__.__name__}")

print("\n--- children() (只一级) ---")
for name, child in net.named_children():
    print(f"  {name:25s}: {child.__class__.__name__}")

# buffers
print("\n--- named_buffers() ---")
for name, buf in net.named_buffers():
    print(f"  {name:25s}: shape={list(buf.shape)}, requires_grad={buf.requires_grad}")

In [ ]:
# train() vs eval() 的影响
print("\n" + "=" * 60)
print("train() vs eval() 的影响")
print("=" * 60)

x = torch.randn(4, 10)  # batch_size=4

# train 模式
net.train()
out_train1 = net(x)
out_train2 = net(x)  # Dropout 使得两次结果不同
print(f"  train 模式: 两次前向一致? {torch.equal(out_train1, out_train2)}  ← Dropout 随机!")

# eval 模式
net.eval()
with torch.no_grad():
    out_eval1 = net(x)
    out_eval2 = net(x)
print(f"  eval 模式: 两次前向一致? {torch.equal(out_eval1, out_eval2)}  ← Dropout 关闭!")

print("\n💡 eval() 影响: Dropout 关闭, BatchNorm 用 running stats")
print("💡 推理时必须: model.eval() + torch.no_grad()")

In [ ]:
# 冻结部分层
print("\n" + "=" * 60)
print("冻结部分层 (Fine-tuning 常用)")
print("=" * 60)

# 冻结 layer1
for param in net.layer1.parameters():
    param.requires_grad = False

trainable_after = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(f"  冻结 layer1 前可训练参数: {trainable:,}")
print(f"  冻结 layer1 后可训练参数: {trainable_after:,}")
print(f"  减少: {trainable - trainable_after:,}")

# 恢复
for param in net.layer1.parameters():
    param.requires_grad = True

---
## 2. Hook 机制 <a id='2'></a>

Hook 让你在不修改模型代码的情况下，**拦截和查看中间层的输入/输出/梯度**。

| Hook 类型 | 注册方法 | 回调签名 | 用途 |
|----------|---------|---------|------|
| Forward Hook | `register_forward_hook` | `hook(module, input, output)` | 查看特征图 |
| Forward Pre-Hook | `register_forward_pre_hook` | `hook(module, input)` | 修改输入 |
| Backward Hook | `register_full_backward_hook` | `hook(module, grad_input, grad_output)` | 查看梯度 |

In [ ]:
print("=" * 60)
print("Forward Hook: 特征图可视化")
print("=" * 60)

# 用一个简单的 CNN
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(32, 10)
    
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = x.flatten(1)
        return self.fc(x)

cnn = SimpleCNN()
features = {}  # 存储中间特征

def get_forward_hook(name):
    def hook(module, input, output):
        features[name] = output.detach()
    return hook

# 注册 hook
hooks = []
hooks.append(cnn.conv1.register_forward_hook(get_forward_hook('conv1')))
hooks.append(cnn.conv2.register_forward_hook(get_forward_hook('conv2')))
hooks.append(cnn.fc.register_forward_hook(get_forward_hook('fc')))

# 前向传播
x = torch.randn(1, 3, 32, 32)
cnn.eval()
with torch.no_grad():
    out = cnn(x)

print("捕获的特征图:")
for name, feat in features.items():
    print(f"  {name:10s}: shape={list(feat.shape)}")

# 可视化 conv1 特征图
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
conv1_feat = features['conv1'][0]  # (16, 32, 32)
for i in range(16):
    ax = axes[i // 8, i % 8]
    ax.imshow(conv1_feat[i].numpy(), cmap='viridis')
    ax.set_title(f'Ch {i}', fontsize=8)
    ax.axis('off')
plt.suptitle('Conv1 特征图 (16 channels)', fontsize=13)
plt.tight_layout()
plt.show()

# 清理 hook
for h in hooks:
    h.remove()

In [ ]:
# Backward Hook: 梯度可视化
print("\n" + "=" * 60)
print("Backward Hook: 梯度流分析")
print("=" * 60)

grad_norms = {}

def get_backward_hook(name):
    def hook(module, grad_input, grad_output):
        # grad_output: 从后面传来的梯度
        if grad_output[0] is not None:
            grad_norms[name] = grad_output[0].norm().item()
    return hook

# 构建深一点的网络
class DeepNet(nn.Module):
    def __init__(self, n_layers=10):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(64, 64) for _ in range(n_layers)
        ])
        self.head = nn.Linear(64, 10)
    
    def forward(self, x):
        for layer in self.layers:
            x = F.relu(layer(x))
        return self.head(x)

deep_net = DeepNet(n_layers=15)

# 注册 backward hook
hooks = []
for i, layer in enumerate(deep_net.layers):
    hooks.append(layer.register_full_backward_hook(get_backward_hook(f'layer_{i}')))

# 前向 + 反向
x = torch.randn(8, 64)
y = torch.randint(0, 10, (8,))
out = deep_net(x)
loss = F.cross_entropy(out, y)
loss.backward()

# 可视化梯度范数
layers = sorted(grad_norms.keys(), key=lambda x: int(x.split('_')[1]))
norms = [grad_norms[l] for l in layers]

plt.figure(figsize=(10, 4))
plt.bar(range(len(norms)), norms, color='steelblue')
plt.xlabel('Layer')
plt.ylabel('Gradient Norm')
plt.title('各层梯度范数 (检测梯度消失/爆炸)')
plt.xticks(range(len(layers)), [f'L{i}' for i in range(len(layers))])
plt.tight_layout()
plt.show()

for h in hooks:
    h.remove()

print("💡 如果梯度范数随层数急剧下降 → 梯度消失")
print("💡 解决方案: ResNet 残差连接 / LayerNorm / 合适的初始化")

---
## 3. state_dict: 模型保存与加载 <a id='3'></a>

In [ ]:
print("=" * 60)
print("state_dict: 模型保存与加载")
print("=" * 60)

model = SimpleCNN()

# 查看 state_dict
print("\n--- state_dict 内容 ---")
for key, val in model.state_dict().items():
    print(f"  {key:30s}: {list(val.shape)}")

# 保存方式 1: 只保存参数 (推荐)
torch.save(model.state_dict(), '/tmp/model_weights.pth')
print(f"\n保存 state_dict → /tmp/model_weights.pth")

# 加载
model2 = SimpleCNN()  # 先创建相同架构的模型
model2.load_state_dict(torch.load('/tmp/model_weights.pth', weights_only=True))
print("加载成功!")

# 验证
x = torch.randn(1, 3, 32, 32)
model.eval()
model2.eval()
with torch.no_grad():
    out1 = model(x)
    out2 = model2(x)
print(f"输出一致: {torch.equal(out1, out2)}")

# 保存方式 2: 保存完整 checkpoint (训练恢复)
optimizer = optim.Adam(model.parameters(), lr=0.001)
checkpoint = {
    'epoch': 10,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': 0.5,
}
torch.save(checkpoint, '/tmp/checkpoint.pth')
print(f"\n保存 checkpoint (含 optimizer 状态) → /tmp/checkpoint.pth")

print("\n💡 只保存 state_dict (不保存整个 model) 更灵活、更安全")
print("💡 训练恢复需要保存 optimizer state + epoch + loss")

---
## 4. 优化器原理: SGD → Momentum → Adam → AdamW <a id='4'></a>

### 4.1 SGD
$$\theta_{t+1} = \theta_t - \eta \cdot g_t$$

### 4.2 SGD + Momentum
$$v_t = \beta v_{t-1} + g_t$$
$$\theta_{t+1} = \theta_t - \eta \cdot v_t$$

### 4.3 Adam (Adaptive Moment Estimation)
$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t \quad \text{(一阶矩: 梯度均值)}$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2 \quad \text{(二阶矩: 梯度方差)}$$
$$\hat{m}_t = m_t / (1-\beta_1^t) \quad \text{(偏差修正)}$$
$$\hat{v}_t = v_t / (1-\beta_2^t)$$
$$\theta_{t+1} = \theta_t - \eta \cdot \hat{m}_t / (\sqrt{\hat{v}_t} + \epsilon)$$

### 4.4 AdamW (Decoupled Weight Decay)
$$\theta_{t+1} = \theta_t - \eta \cdot (\hat{m}_t / (\sqrt{\hat{v}_t} + \epsilon) + \lambda \theta_t)$$

AdamW 把 weight decay 从梯度中解耦出来，效果更好。

In [ ]:
# 手写 Adam
print("=" * 60)
print("手写 Adam 优化器")
print("=" * 60)

class ManualAdam:
    """
    手写 Adam 优化器
    对照公式逐行实现
    """
    def __init__(self, params, lr=0.001, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0):
        self.params = list(params)
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.weight_decay = weight_decay
        self.t = 0  # 时间步
        
        # 为每个参数维护 m 和 v
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
    
    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()
    
    @torch.no_grad()
    def step(self):
        self.t += 1
        for i, p in enumerate(self.params):
            if p.grad is None:
                continue
            
            g = p.grad
            
            # Weight decay (AdamW style)
            if self.weight_decay > 0:
                p.data.mul_(1 - self.lr * self.weight_decay)
            
            # 更新一阶矩 (梯度均值)
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g
            
            # 更新二阶矩 (梯度方差)
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * g ** 2
            
            # 偏差修正
            m_hat = self.m[i] / (1 - self.beta1 ** self.t)
            v_hat = self.v[i] / (1 - self.beta2 ** self.t)
            
            # 更新参数
            p.data -= self.lr * m_hat / (torch.sqrt(v_hat) + self.eps)


# 验证: 手写 Adam vs PyTorch Adam
torch.manual_seed(42)
w_manual = torch.randn(5, 3, requires_grad=True)
w_official = w_manual.data.clone().requires_grad_(True)

opt_manual = ManualAdam([w_manual], lr=0.01)
opt_official = optim.Adam([w_official], lr=0.01)

# 模拟几步训练
for step in range(5):
    # 相同的 loss
    loss_m = (w_manual ** 2).sum()
    loss_o = (w_official ** 2).sum()
    
    opt_manual.zero_grad()
    loss_m.backward()
    opt_manual.step()
    
    opt_official.zero_grad()
    loss_o.backward()
    opt_official.step()

diff = (w_manual.data - w_official.data).abs().max().item()
print(f"  5步后最大差异: {diff:.2e}")
print(f"  一致: {diff < 1e-6}")

In [ ]:
# 偏差修正的重要性
print("\n" + "=" * 60)
print("偏差修正 (Bias Correction) 的影响")
print("=" * 60)

# 模拟: 真实梯度均值=5.0
true_mean = 5.0
beta1 = 0.9

m = 0.0
m_corrected_list = []
m_uncorrected_list = []

for t in range(1, 21):
    g = true_mean + np.random.randn() * 0.5  # 带噪声的梯度
    m = beta1 * m + (1 - beta1) * g
    m_corrected = m / (1 - beta1 ** t)
    
    m_uncorrected_list.append(m)
    m_corrected_list.append(m_corrected)

plt.figure(figsize=(10, 4))
plt.plot(m_uncorrected_list, 'r-o', label='未修正 m', markersize=4)
plt.plot(m_corrected_list, 'b-s', label='修正后 m_hat', markersize=4)
plt.axhline(y=true_mean, color='gray', linestyle='--', label=f'真实均值={true_mean}')
plt.xlabel('Step')
plt.ylabel('Estimated Mean')
plt.title('Adam 偏差修正: 初始阶段差异巨大!')
plt.legend()
plt.tight_layout()
plt.show()

print("💡 初始化 m=0 导致前几步估计严重偏低")
print("💡 除以 (1-β^t) 修正这个偏差")

---
## 5. Adam vs SGD 对比实验 <a id='5'></a>

In [ ]:
# 在 2D 优化问题上可视化轨迹
print("=" * 60)
print("优化器轨迹对比 (2D Rosenbrock 函数)")
print("=" * 60)

def rosenbrock(x, y, a=1, b=100):
    """Rosenbrock 函数: 经典优化测试函数, 最小值在 (a, a²)"""
    return (a - x) ** 2 + b * (y - x ** 2) ** 2

def optimize_trajectory(optimizer_class, lr, n_steps=500, **opt_kwargs):
    x = torch.tensor(-1.5, requires_grad=True)
    y = torch.tensor(1.5, requires_grad=True)
    optimizer = optimizer_class([x, y], lr=lr, **opt_kwargs)
    
    trajectory = [(x.item(), y.item())]
    losses = []
    
    for _ in range(n_steps):
        optimizer.zero_grad()
        loss = rosenbrock(x, y)
        loss.backward()
        optimizer.step()
        trajectory.append((x.item(), y.item()))
        losses.append(loss.item())
    
    return trajectory, losses

# 运行不同优化器
configs = [
    ('SGD', optim.SGD, 0.001, {}),
    ('SGD+Momentum', optim.SGD, 0.001, {'momentum': 0.9}),
    ('Adam', optim.Adam, 0.01, {}),
    ('AdamW', optim.AdamW, 0.01, {'weight_decay': 0.01}),
]

results = {}
for name, cls, lr, kwargs in configs:
    traj, losses = optimize_trajectory(cls, lr, n_steps=500, **kwargs)
    results[name] = (traj, losses)
    final_x, final_y = traj[-1]
    print(f"  {name:15s}: 最终位置=({final_x:.4f}, {final_y:.4f}), "
          f"最终loss={losses[-1]:.6f}")

In [ ]:
# 可视化
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 轨迹图
ax = axes[0]
xx = np.linspace(-2, 2, 200)
yy = np.linspace(-1, 3, 200)
X, Y = np.meshgrid(xx, yy)
Z = (1 - X)**2 + 100*(Y - X**2)**2
ax.contour(X, Y, Z, levels=np.logspace(0, 3.5, 20), cmap='RdYlBu_r', alpha=0.6)
ax.plot(1, 1, 'r*', markersize=15, label='最优解 (1,1)')

colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12']
for (name, (traj, _)), color in zip(results.items(), colors):
    traj = np.array(traj)
    ax.plot(traj[:200, 0], traj[:200, 1], '-', color=color, alpha=0.8, linewidth=1.5, label=name)
    ax.plot(traj[0, 0], traj[0, 1], 'o', color=color, markersize=8)

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('优化轨迹 (Rosenbrock)')
ax.legend(fontsize=9)

# Loss 曲线
ax = axes[1]
for (name, (_, losses)), color in zip(results.items(), colors):
    ax.semilogy(losses, color=color, label=name, linewidth=1.5)
ax.set_xlabel('Step')
ax.set_ylabel('Loss (log scale)')
ax.set_title('Loss 收敛曲线')
ax.legend()

plt.tight_layout()
plt.show()

---
## 6. Dataset 与 DataLoader <a id='6'></a>

### 核心概念
- `Dataset`: 定义数据的存取方式 (`__getitem__`, `__len__`)
- `DataLoader`: 批量加载、打乱、多进程预取

In [ ]:
print("=" * 60)
print("自定义 Dataset")
print("=" * 60)

class SyntheticDataset(Dataset):
    """自定义数据集: 生成合成分类数据"""
    def __init__(self, n_samples=1000, n_features=10, n_classes=3, transform=None):
        self.transform = transform
        
        # 生成数据
        self.X = torch.randn(n_samples, n_features)
        # 简单的线性分类规则
        W = torch.randn(n_features, n_classes)
        logits = self.X @ W
        self.y = logits.argmax(dim=1)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]
        
        if self.transform:
            x = self.transform(x)
        
        return x, y


# 创建
dataset = SyntheticDataset(n_samples=1000)
print(f"  数据集大小: {len(dataset)}")
print(f"  单样本: x.shape={dataset[0][0].shape}, y={dataset[0][1].item()}")

# DataLoader
loader = DataLoader(
    dataset, 
    batch_size=32, 
    shuffle=True,
    num_workers=0,    # 0 = 主进程加载
    pin_memory=True,  # 加速 CPU→GPU 传输
    drop_last=True,   # 丢弃不完整的最后一个 batch
)

# 遍历
for i, (batch_x, batch_y) in enumerate(loader):
    if i == 0:
        print(f"\n  第一个 batch:")
        print(f"    x.shape={batch_x.shape}, y.shape={batch_y.shape}")
        print(f"    y 分布: {torch.bincount(batch_y).tolist()}")

print(f"  总 batch 数: {len(loader)}")

In [ ]:
# DataLoader 性能: num_workers 的影响
print("\n" + "=" * 60)
print("DataLoader: num_workers 性能")
print("=" * 60)

# 模拟慢数据集 (真实场景: 读图片、解码)
class SlowDataset(Dataset):
    def __init__(self, size=500):
        self.size = size
    def __len__(self):
        return self.size
    def __getitem__(self, idx):
        time.sleep(0.001)  # 模拟 1ms IO
        return torch.randn(3, 32, 32), torch.randint(0, 10, (1,)).item()

slow_ds = SlowDataset(200)

for n_workers in [0, 2]:
    loader = DataLoader(slow_ds, batch_size=32, num_workers=n_workers)
    start = time.perf_counter()
    for batch in loader:
        pass  # 只测加载速度
    elapsed = time.perf_counter() - start
    print(f"  num_workers={n_workers}: {elapsed:.2f}s")

print("\n💡 num_workers > 0: 用子进程预取数据，和 GPU 计算重叠")
print("💡 通常设为 CPU 核心数的一半或 4")
print("💡 pin_memory=True: 使用锁页内存，加速 CPU→GPU 传输")

---
## 7. AMP 混合精度训练 <a id='7'></a>

### 7.1 为什么混合精度？

| | FP32 | FP16 |
|---|---|---|
| 内存 | 4 bytes | 2 bytes → **省一半显存** |
| 计算 | 基线 | Tensor Core 加速 → **2-3x 快** |
| 精度 | 高 | 低 (可能溢出) |

### 7.2 AMP 的核心: autocast + GradScaler

- **autocast**: 自动选择哪些运算用 FP16 (如 matmul, conv)，哪些保持 FP32 (如 softmax, loss)
- **GradScaler**: 放大 loss 防止 FP16 梯度下溢 (underflow)

In [ ]:
print("=" * 60)
print("FP16 精度问题演示")
print("=" * 60)

# FP16 的范围和精度
print(f"  FP32 范围: [{torch.finfo(torch.float32).min:.2e}, {torch.finfo(torch.float32).max:.2e}]")
print(f"  FP16 范围: [{torch.finfo(torch.float16).min:.2e}, {torch.finfo(torch.float16).max:.2e}]")
print(f"  FP16 最小正数: {torch.finfo(torch.float16).tiny:.2e}")

# 下溢演示
small_grad = torch.tensor(1e-5, dtype=torch.float32)
print(f"\n  FP32 梯度: {small_grad.item():.2e}")
print(f"  转 FP16:   {small_grad.half().item():.2e}  ← 还能表示")

tiny_grad = torch.tensor(1e-8, dtype=torch.float32)
print(f"\n  FP32 梯度: {tiny_grad.item():.2e}")
print(f"  转 FP16:   {tiny_grad.half().item():.2e}  ← 变成 0! (下溢)")

# GradScaler 的作用
scale = 1024.0
scaled_grad = tiny_grad * scale
print(f"\n  放大 {scale:.0f}x 后 FP16: {scaled_grad.half().item():.2e}  ← 可以表示了!")
print(f"  除回去恢复: {scaled_grad.half().float().item() / scale:.2e}")

print("\n💡 GradScaler 就是做这件事: 放大 loss → 梯度变大 → FP16 不下溢 → 恢复")

In [ ]:
# AMP 基本用法
print("\n" + "=" * 60)
print("AMP 基本用法")
print("=" * 60)

model = SimpleCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scaler = GradScaler(enabled=(device.type == 'cuda'))

# 标准 AMP 训练步骤
x = torch.randn(8, 3, 32, 32).to(device)
y = torch.randint(0, 10, (8,)).to(device)

print("\n--- AMP 训练步骤 ---")

# Step 1: autocast 前向
with autocast(device_type=device.type, enabled=(device.type == 'cuda')):
    logits = model(x)
    loss = F.cross_entropy(logits, y)
    print(f"  1. 前向: logits dtype={logits.dtype}, loss={loss.item():.4f}")

# Step 2: scaled 反向
optimizer.zero_grad()
scaler.scale(loss).backward()
print(f"  2. 反向: scaler.scale={scaler.get_scale():.0f}")

# Step 3: unscale + clip + step
scaler.unscale_(optimizer)  # 恢复梯度
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
scaler.step(optimizer)
scaler.update()  # 动态调整 scale
print(f"  3. 更新: 完成")

print("\n💡 AMP 训练模板:")
print("   with autocast(): loss = model(x)")
print("   scaler.scale(loss).backward()")
print("   scaler.step(optimizer)")
print("   scaler.update()")

---
## 8. 综合实战: AMP 训练 ResNet on CIFAR-10 <a id='8'></a>

完整的训练流程，对比 FP32 vs AMP 的速度和效果。

In [ ]:
# 数据准备
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

full_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
full_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

# 使用子集加速实验
TRAIN_SIZE = 5000
TEST_SIZE = 1000
train_set = Subset(full_train, range(TRAIN_SIZE))
test_set = Subset(full_test, range(TEST_SIZE))

train_loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

print(f"训练集: {len(train_set)}, 测试集: {len(test_set)}")
print(f"Batch 数: train={len(train_loader)}, test={len(test_loader)}")

In [ ]:
# 简化版 ResNet Block
class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
    
    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += residual  # 残差连接
        return F.relu(out)

class MiniResNet(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
        )
        self.layer1 = nn.Sequential(ResBlock(64), ResBlock(64))
        self.layer2 = nn.Sequential(
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            ResBlock(128),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(128, n_classes)
    
    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.pool(x).flatten(1)
        return self.fc(x)

model_test = MiniResNet()
params = sum(p.numel() for p in model_test.parameters())
print(f"MiniResNet 参数量: {params:,} ({params/1e6:.2f}M)")

In [ ]:
# 训练函数
def train_model(model, train_loader, test_loader, optimizer, scheduler=None,
                n_epochs=15, use_amp=False, label=""):
    """通用训练函数"""
    model = model.to(device)
    scaler = GradScaler(enabled=use_amp and device.type == 'cuda')
    criterion = nn.CrossEntropyLoss()
    
    history = {'train_loss': [], 'test_acc': [], 'epoch_time': []}
    
    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0
        n_batches = 0
        start = time.perf_counter()
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            
            with autocast(device_type=device.type, enabled=use_amp and device.type == 'cuda'):
                logits = model(images)
                loss = criterion(logits, labels)
            
            if use_amp and device.type == 'cuda':
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            
            epoch_loss += loss.item()
            n_batches += 1
        
        if scheduler:
            scheduler.step()
        
        epoch_time = time.perf_counter() - start
        avg_loss = epoch_loss / n_batches
        
        # 测试
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                preds = model(images).argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += len(labels)
        acc = correct / total * 100
        
        history['train_loss'].append(avg_loss)
        history['test_acc'].append(acc)
        history['epoch_time'].append(epoch_time)
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  [{label}] Epoch {epoch+1:2d}/{n_epochs} | "
                  f"Loss: {avg_loss:.4f} | Acc: {acc:.1f}% | Time: {epoch_time:.2f}s")
    
    return history

In [ ]:
# 实验 1: SGD vs Adam vs AdamW
print("=" * 60)
print("实验 1: 优化器对比 (CIFAR-10)")
print("=" * 60)

N_EPOCHS = 15
results = {}

# SGD + Momentum
print("\n--- SGD + Momentum ---")
torch.manual_seed(42)
model_sgd = MiniResNet()
opt_sgd = optim.SGD(model_sgd.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
sched_sgd = optim.lr_scheduler.CosineAnnealingLR(opt_sgd, T_max=N_EPOCHS)
results['SGD'] = train_model(model_sgd, train_loader, test_loader, opt_sgd, sched_sgd, N_EPOCHS, label='SGD')

# Adam
print("\n--- Adam ---")
torch.manual_seed(42)
model_adam = MiniResNet()
opt_adam = optim.Adam(model_adam.parameters(), lr=1e-3)
sched_adam = optim.lr_scheduler.CosineAnnealingLR(opt_adam, T_max=N_EPOCHS)
results['Adam'] = train_model(model_adam, train_loader, test_loader, opt_adam, sched_adam, N_EPOCHS, label='Adam')

# AdamW
print("\n--- AdamW ---")
torch.manual_seed(42)
model_adamw = MiniResNet()
opt_adamw = optim.AdamW(model_adamw.parameters(), lr=1e-3, weight_decay=0.01)
sched_adamw = optim.lr_scheduler.CosineAnnealingLR(opt_adamw, T_max=N_EPOCHS)
results['AdamW'] = train_model(model_adamw, train_loader, test_loader, opt_adamw, sched_adamw, N_EPOCHS, label='AdamW')

In [ ]:
# 实验 2: FP32 vs AMP (如果有 GPU)
if device.type == 'cuda':
    print("\n" + "=" * 60)
    print("实验 2: FP32 vs AMP")
    print("=" * 60)
    
    # FP32
    print("\n--- FP32 ---")
    torch.manual_seed(42)
    model_fp32 = MiniResNet()
    opt_fp32 = optim.AdamW(model_fp32.parameters(), lr=1e-3, weight_decay=0.01)
    sched_fp32 = optim.lr_scheduler.CosineAnnealingLR(opt_fp32, T_max=N_EPOCHS)
    results['FP32'] = train_model(model_fp32, train_loader, test_loader, opt_fp32, sched_fp32, N_EPOCHS, use_amp=False, label='FP32')
    
    # AMP
    print("\n--- AMP ---")
    torch.manual_seed(42)
    model_amp = MiniResNet()
    opt_amp = optim.AdamW(model_amp.parameters(), lr=1e-3, weight_decay=0.01)
    sched_amp = optim.lr_scheduler.CosineAnnealingLR(opt_amp, T_max=N_EPOCHS)
    results['AMP'] = train_model(model_amp, train_loader, test_loader, opt_amp, sched_amp, N_EPOCHS, use_amp=True, label='AMP')
    
    fp32_time = sum(results['FP32']['epoch_time'])
    amp_time = sum(results['AMP']['epoch_time'])
    print(f"\n  FP32 总时间: {fp32_time:.1f}s")
    print(f"  AMP 总时间:  {amp_time:.1f}s")
    print(f"  加速比: {fp32_time/amp_time:.2f}x")
else:
    print("\n⚠️ 无 GPU，跳过 AMP 对比 (AMP 需要 CUDA)")

In [ ]:
# 可视化结果
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'SGD': '#e74c3c', 'Adam': '#3498db', 'AdamW': '#2ecc71', 'FP32': '#f39c12', 'AMP': '#9b59b6'}

for name, hist in results.items():
    c = colors.get(name, 'gray')
    axes[0].plot(hist['train_loss'], color=c, label=name, linewidth=2)
    axes[1].plot(hist['test_acc'], color=c, label=name, linewidth=2)

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Train Loss')
axes[0].set_title('训练 Loss')
axes[0].legend()

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test Accuracy (%)')
axes[1].set_title('测试准确率')
axes[1].legend()

plt.tight_layout()
plt.show()

# 汇总表
print("\n" + "=" * 60)
print(f"{'方法':10s} | {'最终Acc':>8s} | {'最佳Acc':>8s} | {'总时间':>8s}")
print("-" * 45)
for name, hist in results.items():
    final = hist['test_acc'][-1]
    best = max(hist['test_acc'])
    total_t = sum(hist['epoch_time'])
    print(f"{name:10s} | {final:>7.1f}% | {best:>7.1f}% | {total_t:>6.1f}s")

---
## 9. 学习率调度器 <a id='9'></a>

In [ ]:
print("=" * 60)
print("常用学习率调度器")
print("=" * 60)

def plot_scheduler(scheduler_name, scheduler_fn, n_epochs=100):
    model = nn.Linear(10, 1)
    optimizer = optim.SGD(model.parameters(), lr=0.1)
    scheduler = scheduler_fn(optimizer)
    
    lrs = []
    for _ in range(n_epochs):
        lrs.append(optimizer.param_groups[0]['lr'])
        optimizer.step()
        scheduler.step()
    return lrs

schedulers = {
    'StepLR(30, γ=0.1)': lambda opt: optim.lr_scheduler.StepLR(opt, step_size=30, gamma=0.1),
    'CosineAnnealing': lambda opt: optim.lr_scheduler.CosineAnnealingLR(opt, T_max=100),
    'ExponentialLR(γ=0.95)': lambda opt: optim.lr_scheduler.ExponentialLR(opt, gamma=0.95),
    'OneCycleLR': lambda opt: optim.lr_scheduler.OneCycleLR(opt, max_lr=0.1, total_steps=100),
}

fig, ax = plt.subplots(figsize=(12, 5))
for name, fn in schedulers.items():
    lrs = plot_scheduler(name, fn)
    ax.plot(lrs, label=name, linewidth=2)

ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_title('学习率调度器对比')
ax.legend()
plt.tight_layout()
plt.show()

print("💡 CosineAnnealing: 最常用, 平滑衰减")
print("💡 OneCycleLR: Warmup + 衰减, Super-Convergence")
print("💡 Warmup + CosineAnnealing: Transformer 标配")

---
## 10. 思考题 & 总结 <a id='10'></a>

### 今日核心知识点

1. **nn.Module**: parameters / modules / train/eval / 冻结层 / buffer
2. **Hook**: forward_hook 查看特征图, backward_hook 分析梯度流
3. **state_dict**: 保存/加载权重, checkpoint 恢复训练
4. **Adam**: 一阶矩+二阶矩+偏差修正; AdamW 解耦权重衰减
5. **DataLoader**: batch/shuffle/num_workers/pin_memory
6. **AMP**: autocast (自动选精度) + GradScaler (防下溢)
7. **LR Scheduler**: Step/Cosine/OneCycle/Warmup

### 面试高频问题

1. **Adam 和 SGD 的区别？什么时候用哪个？** → Adam 收敛快但泛化可能差; SGD+Momentum 泛化好
2. **AdamW 和 Adam+L2 的区别？** → AdamW 解耦 weight decay, 效果更好
3. **train() 和 eval() 分别影响什么？** → Dropout, BatchNorm
4. **AMP 的原理？为什么需要 GradScaler？** → FP16 下溢; 放大梯度
5. **num_workers 怎么设？** → CPU 核数一半或 4, 需要实测

### 明日预习: micrograd 引擎
- Karpathy 的 micrograd
- Value 类 / 计算图 / 反向传播 / 优化器

In [ ]:
print("\n" + "=" * 60)
print("W1 Day4 完成! 🎉")
print("=" * 60)
print("""
今日成果:
  ✅ nn.Module 深入: parameters/modules/train/eval/冻结/buffer
  ✅ Hook 机制: forward_hook 特征图可视化 + backward_hook 梯度流分析
  ✅ state_dict 保存/加载 + checkpoint 恢复
  ✅ 手写 Adam 优化器 (与 PyTorch 验证一致)
  ✅ 偏差修正 (Bias Correction) 可视化
  ✅ 优化器轨迹对比 (Rosenbrock 函数: SGD/Momentum/Adam/AdamW)
  ✅ 自定义 Dataset + DataLoader (num_workers 性能)
  ✅ AMP 混合精度: FP16 下溢演示 + GradScaler 原理
  ✅ 综合实战: MiniResNet on CIFAR-10 (SGD vs Adam vs AdamW + FP32 vs AMP)
  ✅ 学习率调度器对比 (Step/Cosine/Exponential/OneCycle)
""")